# Ray Data Processing on RHOAI

This notebook demonstrates distributed data processing with Ray Data on an RHOAI KubeRay cluster.

**What you will learn:**
- Create a dataset with `ray.data.from_items()`
- Transform data with `map()` and `filter()`
- Aggregate results across distributed workers

**Prerequisites:**
- Running in an RHOAI Workbench
- A RayCluster running in `ray-demo` namespace

## RHOAI 3.4.1: AuthenticationReady Workaround

> **Important:** On RHOAI 3.4.1, after creating a cluster, a cluster administrator must run:
> ```bash
> ./scripts/fix-auth.sh ray-demo <cluster-name>
> ```
> before `cluster.wait_ready()` will succeed.

In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

In [ ]:
cluster = Cluster(
    ClusterConfiguration(
        name="data-processing",
        namespace="ray-demo",
        num_workers=2,
        worker_cpu_requests="100m",
        worker_memory_requests=2,
        image="quay.io/modh/ray:2.47.1-py311-cu121",
        local_queue="default",
    )
)
cluster.apply()
# Admin: run ./scripts/fix-auth.sh ray-demo data-processing
cluster.wait_ready()

## Connect with mTLS

In [ ]:
from codeflare_sdk import generate_cert
generate_cert.generate_tls_cert(cluster.config.name, cluster.config.namespace)
generate_cert.export_env(cluster.config.name, cluster.config.namespace)

import ray
ray.init(cluster.cluster_uri())
print(f"Connected to {len(ray.nodes())} nodes")

## Create a Dataset

In [ ]:
# Create a dataset of records
records = [
    {"name": "Alice", "age": 30, "score": 85.5},
    {"name": "Bob", "age": 25, "score": 92.0},
    {"name": "Charlie", "age": 35, "score": 78.3},
    {"name": "Diana", "age": 28, "score": 95.1},
    {"name": "Eve", "age": 22, "score": 88.7},
    {"name": "Frank", "age": 40, "score": 71.2},
    {"name": "Grace", "age": 33, "score": 91.8},
    {"name": "Hank", "age": 27, "score": 82.4},
]

ds = ray.data.from_items(records)
print(f"Dataset: {ds.count()} records")
ds.show(3)

## Transform: Normalize Scores

In [ ]:
def normalize_score(record):
    record["grade"] = "A" if record["score"] >= 90 else "B" if record["score"] >= 80 else "C"
    record["score_pct"] = round(record["score"] / 100.0, 3)
    return record

graded = ds.map(normalize_score)
graded.show()

## Filter: High Performers

In [ ]:
high_performers = graded.filter(lambda r: r["grade"] == "A")
print(f"High performers (grade A): {high_performers.count()}")
high_performers.show()

## Aggregate: Statistics

In [ ]:
import numpy as np

scores = [r["score"] for r in ds.take_all()]
print(f"Mean score: {np.mean(scores):.2f}")
print(f"Std dev:    {np.std(scores):.2f}")
print(f"Min:        {np.min(scores):.2f}")
print(f"Max:        {np.max(scores):.2f}")

## Cleanup

In [ ]:
ray.shutdown()
cluster.down()
print("Cluster deleted.")